## 02 - Camada Bronze

 ## MVP-3 — Engenharia de Dados

 Este notebook cria a camada Bronze a partir dos arquivos
 originais armazenados na Landing Zone.

 ### Fontes
 - IMDb Dataset of 50K Movie Reviews
 - Rotten Tomatoes Movies
 - Rotten Tomatoes Critic Reviews

 ### Fluxo
 Landing Zone → leitura dos CSVs → metadados técnicos → tabelas Delta Bronze

 ### Princípios da camada Bronze
 - preservar os dados originais;
 - não realizar limpeza de dados;
 - não remover registros;
 - não eliminar duplicidades;

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    col,
    lit
)


# ============================================================
# CATÁLOGO E SCHEMAS
# ============================================================

CATALOGO = "mvp3_eng_dados"

SCHEMA_DADOS_BRUTOS = "dados_brutos"
SCHEMA_BRONZE = "bronze"


# ============================================================
# LANDING ZONE
# ============================================================

CAMINHO_VOLUME = (
    f"/Volumes/{CATALOGO}/"
    f"{SCHEMA_DADOS_BRUTOS}/"
    "volume_dados_brutos"
)

CAMINHO_IMDB = (
    f"{CAMINHO_VOLUME}/imdb/IMDB Dataset.csv"
)

CAMINHO_RT_FILMES = (
    f"{CAMINHO_VOLUME}/rotten_tomatoes/"
    "rotten_tomatoes_movies.csv"
)

CAMINHO_RT_AVALIACOES = (
    f"{CAMINHO_VOLUME}/rotten_tomatoes/"
    "rotten_tomatoes_critic_reviews.csv"
)


# ============================================================
# TABELAS BRONZE
# ============================================================

TABELA_IMDB = (
    f"{CATALOGO}.{SCHEMA_BRONZE}.imdb_avaliacoes"
)

TABELA_RT_FILMES = (
    f"{CATALOGO}.{SCHEMA_BRONZE}.rt_filmes"
)

TABELA_RT_AVALIACOES = (
    f"{CATALOGO}.{SCHEMA_BRONZE}.rt_avaliacoes"
)


print("✓ Configuração carregada")

✓ Configuração carregada


In [0]:
def ler_csv(caminho):
    return (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(caminho)
    )


df_imdb = ler_csv(
    CAMINHO_IMDB
)

df_rt_filmes = ler_csv(
    CAMINHO_RT_FILMES
)

df_rt_avaliacoes = ler_csv(
    CAMINHO_RT_AVALIACOES
)


print("✓ IMDb carregado")
print("✓ RT Filmes carregado")
print("✓ RT Avaliações carregado")

✓ IMDb carregado
✓ RT Filmes carregado
✓ RT Avaliações carregado


In [0]:

def adicionar_metadados(df, fonte):

    return (
        df
        .withColumn(
            "_fonte",
            lit(fonte)
        )
        .withColumn(
            "_arquivo_origem",
            col("_metadata.file_path")
        )
        .withColumn(
            "_data_ingestao",
            current_timestamp()
        )
    )


df_imdb_bronze = adicionar_metadados(
    df_imdb,
    "IMDb"
)

df_rt_filmes_bronze = adicionar_metadados(
    df_rt_filmes,
    "Rotten Tomatoes"
)

df_rt_avaliacoes_bronze = adicionar_metadados(
    df_rt_avaliacoes,
    "Rotten Tomatoes"
)


print("✓ Metadados técnicos adicionados")

###----

✓ Metadados técnicos adicionados


In [0]:
print("=" * 60)
print("IMDb")
print("=" * 60)

df_imdb_bronze.printSchema()


print("\n" + "=" * 60)
print("ROTTEN TOMATOES - FILMES")
print("=" * 60)

df_rt_filmes_bronze.printSchema()


print("\n" + "=" * 60)
print("ROTTEN TOMATOES - AVALIAÇÕES")
print("=" * 60)

df_rt_avaliacoes_bronze.printSchema()

IMDb
root
 |-- review: string (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- _fonte: string (nullable = false)
 |-- _arquivo_origem: string (nullable = false)
 |-- _data_ingestao: timestamp (nullable = false)


ROTTEN TOMATOES - FILMES
root
 |-- rotten_tomatoes_link: string (nullable = true)
 |-- movie_title: string (nullable = true)
 |-- movie_info: string (nullable = true)
 |-- critics_consensus: string (nullable = true)
 |-- content_rating: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- directors: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- actors: string (nullable = true)
 |-- original_release_date: string (nullable = true)
 |-- streaming_release_date: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- production_company: string (nullable = true)
 |-- tomatometer_status: string (nullable = true)
 |-- tomatometer_rating: string (nullable = true)
 |-- tomatometer_count: string (nullable = true)
 |-- audi

In [0]:
# ============================================================
# IMDb
# ============================================================

(
    df_imdb_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABELA_IMDB)
)


# ============================================================
# ROTTEN TOMATOES - FILMES
# ============================================================

(
    df_rt_filmes_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABELA_RT_FILMES)
)


# ============================================================
# ROTTEN TOMATOES - AVALIAÇÕES
# ============================================================

(
    df_rt_avaliacoes_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABELA_RT_AVALIACOES)
)


print("✓ Tabelas Bronze persistidas")

✓ Tabelas Bronze persistidas


In [0]:
tabelas_bronze = [
    TABELA_IMDB,
    TABELA_RT_FILMES,
    TABELA_RT_AVALIACOES
]


for tabela in tabelas_bronze:

    assert spark.catalog.tableExists(tabela), (
        f"Tabela não encontrada: {tabela}"
    )

    print(f"✓ {tabela}")

✓ mvp3_eng_dados.bronze.imdb_avaliacoes
✓ mvp3_eng_dados.bronze.rt_filmes
✓ mvp3_eng_dados.bronze.rt_avaliacoes


In [0]:
# ============================================================
# CONTAGENS DA ORIGEM
# ============================================================

qtd_imdb_origem = df_imdb.count()

qtd_rt_filmes_origem = df_rt_filmes.count()

qtd_rt_avaliacoes_origem = (
    df_rt_avaliacoes.count()
)


# ============================================================
# CONTAGENS DA BRONZE
# ============================================================

qtd_imdb_bronze = spark.table(
    TABELA_IMDB
).count()

qtd_rt_filmes_bronze = spark.table(
    TABELA_RT_FILMES
).count()

qtd_rt_avaliacoes_bronze = spark.table(
    TABELA_RT_AVALIACOES
).count()


print(
    f"IMDb: "
    f"origem={qtd_imdb_origem:,} | "
    f"bronze={qtd_imdb_bronze:,}"
)

print(
    f"RT Filmes: "
    f"origem={qtd_rt_filmes_origem:,} | "
    f"bronze={qtd_rt_filmes_bronze:,}"
)

print(
    f"RT Avaliações: "
    f"origem={qtd_rt_avaliacoes_origem:,} | "
    f"bronze={qtd_rt_avaliacoes_bronze:,}"
)

IMDb: origem=50,000 | bronze=50,000
RT Filmes: origem=17,712 | bronze=17,712
RT Avaliações: origem=1,130,017 | bronze=1,130,017


In [0]:
assert qtd_imdb_origem == qtd_imdb_bronze, (
    "IMDb: quantidade de registros divergente."
)

assert qtd_rt_filmes_origem == qtd_rt_filmes_bronze, (
    "RT Filmes: quantidade de registros divergente."
)

assert qtd_rt_avaliacoes_origem == qtd_rt_avaliacoes_bronze, (
    "RT Avaliações: quantidade de registros divergente."
)


print("✓ Quantidade de registros preservada")

✓ Quantidade de registros preservada


In [0]:
def validar_colunas_origem(
    df_origem,
    tabela_bronze
):

    colunas_origem = set(
        df_origem.columns
    )

    colunas_bronze = set(
        spark.table(tabela_bronze).columns
    )

    assert colunas_origem.issubset(
        colunas_bronze
    ), (
        f"Colunas da origem não preservadas em "
        f"{tabela_bronze}"
    )


validar_colunas_origem(
    df_imdb,
    TABELA_IMDB
)

validar_colunas_origem(
    df_rt_filmes,
    TABELA_RT_FILMES
)

validar_colunas_origem(
    df_rt_avaliacoes,
    TABELA_RT_AVALIACOES
)


print("✓ Colunas originais preservadas")

✓ Colunas originais preservadas


In [0]:
print("IMDb Bronze")

display(
    spark.table(
        TABELA_IMDB
    ).limit(5)
)


print("Rotten Tomatoes - Filmes Bronze")

display(
    spark.table(
        TABELA_RT_FILMES
    ).limit(5)
)


print("Rotten Tomatoes - Avaliações Bronze")

display(
    spark.table(
        TABELA_RT_AVALIACOES
    ).limit(5)
)

IMDb Bronze


review,sentiment,_fonte,_arquivo_origem,_data_ingestao
"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences, forget charm, forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal, I couldn't say I was ready for it, but as I watched more, I developed a taste for Oz, and got accustomed to the high levels of graphic violence. Not just violence, but injustice (crooked guards who'll be sold out for a nickel, inmates who'll kill on order and get away with it, well mannered, middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz, you may become comfortable with what is uncomfortable viewing....thats if you can get in touch with your darker side.",positive,IMDb,dbfs:/Volumes/mvp3_eng_dados/dados_brutos/volume_dados_brutos/imdb/IMDB%20Dataset.csv,2026-09-27T18:39:42.991Z
"A wonderful little production. The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. The actors are extremely well chosen- Michael Sheen not only ""has got all the polari"" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master's of comedy and his life. The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional 'dream' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell's murals decorating every surface) are terribly well done.",positive,IMDb,dbfs:/Volumes/mvp3_eng_dados/dados_brutos/volume_dados_brutos/imdb/IMDB%20Dataset.csv,2026-09-27T18:39:42.991Z
"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic, but the dialogue is witty and the characters are likable (even the well bread suspected serial killer). While some may be disappointed when they realize this is not Match Point 2: Risk Addiction, I thought it was proof that Woody Allen is still fully in control of the style many of us have grown to love.This was the most I'd laughed at one of Woody's comedies in years (dare I say a decade?). While I've never been impressed with Scarlet Johanson, in this she managed to tone down her ""sexy"" image and jumped right into a average, but spirited young woman.This may not be the crown jewel of his career, but it was wittier than ""Devil Wears Prada"" and more interesting than ""Superman"" a great comedy to go see with friends.",positive,IMDb,dbfs:/Volumes/mvp3_eng_dados/dados_brutos/volume_dados_brutos/imdb/IMDB%20D

Rotten Tomatoes - Filmes Bronze


rotten_tomatoes_link,movie_title,movie_info,critics_consensus,content_rating,genres,directors,authors,actors,original_release_date,streaming_release_date,runtime,production_company,tomatometer_status,tomatometer_rating,tomatometer_count,audience_status,audience_rating,audience_count,tomatometer_top_critics_count,tomatometer_fresh_critics_count,tomatometer_rotten_critics_count,_fonte,_arquivo_origem,_data_ingestao
m/0814255,Percy Jackson & the Olympians: The Lightning Thief,"Always trouble-prone, the life of teenager Percy Jackson (Logan Lerman) gets a lot more complicated when he learns he's the son of the Greek god Poseidon. At a training ground for the children of deities, Percy learns to harness his divine powers and prepare for the adventure of a lifetime: he must prevent a feud among the Olympians from erupting into a devastating war on Earth, and rescue his mother from the clutches of Hades, god of the underworld.","Though it may seem like just another Harry Potter knockoff, Percy Jackson benefits from a strong supporting cast, a speedy plot, and plenty of fun with Greek mythology.",PG,"Action & Adventure, Comedy, Drama, Science Fiction & Fantasy",Chris Columbus,"Craig Titley, Chris Columbus, Rick Riordan","Logan Lerman, Brandon T. Jackson, Alexandra Daddario, Jake Abel, Sean Bean, Pierce Brosnan, Steve Coogan, Rosario Dawson, Melina Kanakaredes, Catherine Keener, Kevin Mckidd, Joe Pantoliano, Uma Thurman, Ray Winstone, Julian Richings, Bonita Friedericy, Annie Ilonzeh, Tania Saulnier, Marie Avgeropoulos, Luisa D'Oliveira, Christie Laing, Marielle Jaffe, Elisa King, Chrystal Tisiga, Alexis Knapp, Charlie Gallant, Chelan Simmons, Andrea Brooks, Natassia Malthe, Max Van Ville, Serinda Swan, Dimitri Lekkos, Ona Grauer, Stefanie von Pfetten, Conrad Coates, Erica Cerra, Dylan Neal, Luke Camilleri, Holly Hougham, Ina Geraldine, Raquel Riskin, Yusleidis Oquendo, Janine Edwards, Valerie Tian, Violet Columbus, Sarah Smyth, Merritt Patterson, Julie Luck, Andrea Day, John Stewart, Suzanne Ristic, Deejay Jackson, Matthew Garrick, Stan Carp, Suzanna Ristic, Richard Harmon, Maria Olsen, Robin Lemon, Doyle Devereux, Tom Pickett, VJ Delos-Reyes, Tim Aas, Keith Dallas, Spencer Atkinson, Maya Washington, Loyd Bateman, Victor Ayala, Zane Holtz, Eli Zagoudakis, Matt Reimer, Rob Hayter, Lloyd Bateman, Shawn Beaton, Jarod Joseph, Reilly Dolman, Paul Cummings, Julie Brar, Dejan Loyola, Damian Arman, Mario Casoria, Dorla Bell, Carolyn Adair (II), Jade Pawluk, G. Patrick Currie, Darian Arman, Mariela Zapata, David L. Smith",2010-02-12,2015-11-25,119,20th Century Fox,Rotten,49,149,Spilled,53,254421,43,73,76,Rotten Tomatoes,dbfs:/Volumes/mvp3_eng_dados/dados_brutos/volume_dados_brutos/rotten_tomatoes/rotten_tomatoes_movies.csv,2026-09-27T18:39:50.896Z
m/0878835,Please Give,"Kate (Catherine Keener) and her husband Alex (Oliver Platt) are wealthy New Yorkers who prowl estate sales and make a tidy profit reselling items they bought cheaply. They buy the apartment next door and plan to remodel just as soon as its current occupant, a cranky old woman, dies. Kate is troubled by the way she and her husband earn a living, and tries to assuage her guilt by befriending her tenant and the woman's granddaughters, but her overtures lead to unexpected consequences.","Nicole Holofcener's newest might seem slight in places, but its rendering of complex characters in a conflicted economic landscape is varied, natural, and touching all the same.",R,Comedy,Nicole Holofcener,Nicole Holofcener,"Catherine Keener, Amanda Peet, Oliver Platt, Rebecca Hall, Sarah Steele, Ann Morgan Guilbert, Kevin Corrigan, Lois Smith",2010-04-30,2012-09-04,90,Sony Pictures Classics,Certified-Fresh,87,142,Upright,64,11574,44,123,19,Rotten Tomatoes,dbfs:/Volumes/mvp3_eng_dados/dados_brutos/volume_dados_brutos/rotten_tomatoes/rotten_tomatoes_movies.csv,2026-09-27T18:39:50.896Z
m/10,10,"A successful, middle-aged Hollywood songwriter falls hopelessly in love with the woman of his dreams, and e

Rotten Tomatoes - Avaliações Bronze


rotten_tomatoes_link,critic_name,top_critic,publisher_name,review_type,review_score,review_date,review_content,_fonte,_arquivo_origem,_data_ingestao
m/0814255,Andrew L. Urban,False,Urban Cinefile,Fresh,null,2010-02-06,A fantasy adventure that fuses Greek mythology to contemporary American places and values. Anyone around 15 (give or take a couple of years) will thrill to the visual spectacle,Rotten Tomatoes,dbfs:/Volumes/mvp3_eng_dados/dados_brutos/volume_dados_brutos/rotten_tomatoes/rotten_tomatoes_critic_reviews.csv,2026-09-27T18:39:55.532Z
m/0814255,Louise Keller,False,Urban Cinefile,Fresh,null,2010-02-06,"Uma Thurman as Medusa, the gorgon with a coiffure of writhing snakes and stone-inducing hypnotic gaze is one of the highlights of this bewitching fantasy",Rotten Tomatoes,dbfs:/Volumes/mvp3_eng_dados/dados_brutos/volume_dados_brutos/rotten_tomatoes/rotten_tomatoes_critic_reviews.csv,2026-09-27T18:39:55.532Z
m/0814255,null,False,FILMINK (Australia),Fresh,null,2010-02-09,"With a top-notch cast and dazzling special effects, this will tide the teens over until the next Harry Potter instalment.",Rotten Tomatoes,dbfs:/Volumes/mvp3_eng_dados/dados_brutos/volume_dados_brutos/rotten_tomatoes/rotten_tomatoes_critic_reviews.csv,2026-09-27T18:39:55.532Z
m/0814255,Ben McEachen,False,Sunday Mail (Australia),Fresh,3.5/5,2010-02-09,"Whether audiences will get behind The Lightning Thief is hard to predict. Overall, it's an entertaining introduction to a promising new world -- but will the consuming shadow of Potter be too big to break free of?",Rotten Tomatoes,dbfs:/Volumes/mvp3_eng_dados/dados_brutos/volume_dados_brutos/rotten_tomatoes/rotten_tomatoes_critic_reviews.csv,2026-09-27T18:39:55.532Z
m/0814255,Ethan Alter,True,Hollywood Reporter,Rotten,null,2010-02-10,"What's really lacking in The Lightning Thief is a genuine sense of wonder, the same thing that brings viewers back to Hogwarts over and over again.",Rotten Tomatoes,dbfs:/Volumes/mvp3_eng_dados/dados_brutos/volume_dados_brutos/rotten_tomatoes/rotten_tomatoes_critic_reviews.csv,2026-09-27T18:39:55.532Z


In [0]:
print("=" * 60)
print("CAMADA BRONZE CONCLUÍDA")
print("=" * 60)


print("\nTabelas criadas:")

print(
    f"✓ {TABELA_IMDB}"
)

print(
    f"✓ {TABELA_RT_FILMES}"
)

print(
    f"✓ {TABELA_RT_AVALIACOES}"
)


print("\nValidações:")

print("✓ Dados originais preservados")
print("✓ Quantidade de registros preservada")
print("✓ Colunas originais preservadas")
print("✓ Metadados técnicos adicionados")
print("✓ Tabelas Delta persistidas")


print("\nTratamentos realizados:")

print("NENHUM")


print("\nStatus:")

print("✓ CAMADA BRONZE VALIDADA")


print("\nPróxima etapa:")

print("03_profiling_qualidade_bronze")

CAMADA BRONZE CONCLUÍDA

Tabelas criadas:
✓ mvp3_eng_dados.bronze.imdb_avaliacoes
✓ mvp3_eng_dados.bronze.rt_filmes
✓ mvp3_eng_dados.bronze.rt_avaliacoes

Validações:
✓ Dados originais preservados
✓ Quantidade de registros preservada
✓ Colunas originais preservadas
✓ Metadados técnicos adicionados
✓ Tabelas Delta persistidas

Tratamentos realizados:
NENHUM

Status:
✓ CAMADA BRONZE VALIDADA

Próxima etapa:
03_profiling_qualidade_bronze
